# まだ試してない改善のうち「劇的に上がりそう」なものだけ試す

**参照**: `docs/10_FIRST_PLACE_AND_IMPROVEMENT_REMAINING.md` §3・§2 の「まだ試していない」かつスコアが大きく伸びる可能性がある項目のみ。

**ベースライン**: Public **0.76479**（55 特徴 + BPR 64。`submission_2hop_bpr64_only.csv`）。  
→ すでに 1 回出しているなら**再提出不要**。このノートで bpr64_only を「出す」のは**ブレンド用**（ALS64 / BPR128 と均等平均するときに読むため）だけ。

**2-hop について**: ファイル名の `submission_2hop_*` は lib のモジュール名。**bpr64_only / bpr128_only は 2-hop 列を足していない**（55 + BPR のみ）。2-hop 特徴を足しているのは #2（movie_review_count 1 列）と #3（割り算+count1）だけ。

| # | 実験内容 | 提出ファイル | 根拠 |
|---|----------|-------------|------|
| 1 | ALS 64 単体 | submission_atmacup_implicit_als64.csv | BPR と目的関数が違うので補正になりうる（09） |
| 2 | BPR 64 + movie_review_count 1列 | submission_2hop_bpr64_count1.csv | 2-hop のうちリークなしの 1 列だけ（09） |
| 3 | 割り算 + 2-hop 1列の組み合わせ | submission_2hop_bpr64_ratio_count1.csv | 1位で割り算と2-hopの組み合わせが効いた（10 §2.2） |
| 4 | BPR 64 と ALS 64 のブレンド（均等） | submission_blend_bpr64_als64.csv | 行列分解の違いで安定・微増の可能性（09） |
| 5 | BPR 64 と BPR 128 のブレンド（均等） | submission_blend_bpr64_bpr128.csv | 同上 |
| 6 | BPR 64 ベース LGB+XGB+CatBoost→Ridge | submission_improvement_03_stacking.csv | 07で55特徴ベースは実装済み。BPR64ベースでは未実施（10 §2.3） |

**前提**: ブレンド用に `submission_2hop_bpr64_only.csv` がローカルにある必要がある。先に `train_baseline.ipynb` で作っておくか、下記「1. ベースライン提出」で 1 本出す（その CSV を Kaggle に再提出する必要はない）。

## パッケージのインストール（初回のみ）

以下を実行すると `requirements.txt` から依存パッケージをインストールします。**プロジェクトルート（ds_dojo4）でノートを開いている場合**に有効です。既にインストール済みならスキップしてよい。

In [1]:
# プロジェクトルート（ds_dojo4）でノートを開いている前提。初回のみ実行すればよい。
%pip install -q -r requirements.txt
print("パッケージ確認済み（requirements.txt）。")

Note: you may need to restart the kernel to use updated packages.
パッケージ確認済み（requirements.txt）。


## セットアップ

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from lib.improvement_candidates import get_setup, run_atmacup_implicit
from lib.two_hop import run_experiment, TWO_HOP_REVIEW_COUNT
from lib.submission import save_submission, verify_submission

ctx = get_setup(seed=42, output_dir="outputs", use_best_pipeline=True)
print(f"Train: {len(ctx.train):,}, Test: {len(ctx.test):,}, Features: {len(ctx.features)}")
print(f"提出先: {ctx.submissions_dir}")

Train: 653,507, Test: 40,716, Features: 55
提出先: outputs/submissions


## 1. ベースライン提出（bpr64_only）・ALS 64

In [3]:
# ブレンド用に bpr64_only を用意（既に存在する場合はスキップ可）
path_bpr64 = ctx.submissions_dir / "submission_2hop_bpr64_only.csv"
if not path_bpr64.exists():
    r = run_experiment(ctx, "bpr64_only", bpr_factors=64)
    print(f"  [bpr64_only] → {path_bpr64.name}  ({'OK' if r.get('ok') else r.get('message')})")
else:
    print(f"  [bpr64_only] 既存ファイルを使用: {path_bpr64.name}")

# 1. ALS 64 単体
r_als = run_atmacup_implicit(ctx, "als", 64, "als64")
print(f"  [ALS 64] → submission_atmacup_implicit_als64.csv  ({'OK' if r_als.get('ok') else r_als.get('message')})")

  0%|          | 0/100 [00:00<?, ?it/s]

  [bpr64_only] → submission_2hop_bpr64_only.csv  (OK)


  0%|          | 0/15 [00:00<?, ?it/s]

  [ALS 64] → submission_atmacup_implicit_als64.csv  (OK)


## 2. BPR 64 + movie_review_count 1列 / 3. 割り算 + 2-hop 1列

In [4]:
# 2. BPR 64 + movie_review_count 1列のみ（リークなし）
path_count1 = ctx.submissions_dir / "submission_2hop_bpr64_count1.csv"
if not path_count1.exists():
    r_count = run_experiment(ctx, "bpr64_count1", bpr_factors=64, use_2hop_cols=[TWO_HOP_REVIEW_COUNT])
    print(f"  [BPR64+count1] → submission_2hop_bpr64_count1.csv  ({'OK' if r_count.get('ok') else r_count.get('message')})")
else:
    print(f"  [BPR64+count1] 既存ファイルを使用: {path_count1.name}")

# 3. 割り算 + 2-hop 1列の組み合わせ（1位で効いたパターン）
path_ratio_count1 = ctx.submissions_dir / "submission_2hop_bpr64_ratio_count1.csv"
if not path_ratio_count1.exists():
    r_ratio = run_experiment(ctx, "bpr64_ratio_count1", bpr_factors=64, use_2hop_cols=[TWO_HOP_REVIEW_COUNT], use_2hop_ratio=True)
    print(f"  [BPR64+ratio+count1] → submission_2hop_bpr64_ratio_count1.csv  ({'OK' if r_ratio.get('ok') else r_ratio.get('message')})")
else:
    print(f"  [BPR64+ratio+count1] 既存ファイルを使用: {path_ratio_count1.name}")

  0%|          | 0/100 [00:00<?, ?it/s]

  [BPR64+count1] → submission_2hop_bpr64_count1.csv  (OK)


  0%|          | 0/100 [00:00<?, ?it/s]

  [BPR64+ratio+count1] → submission_2hop_bpr64_ratio_count1.csv  (OK)


## 4. BPR 128 単体（ブレンド用）・5. ブレンド 2 種

In [6]:
# BPR 128 単体（ブレンド用）
r_bpr128 = run_experiment(ctx, "bpr128_only", bpr_factors=128)
print(f"  [BPR 128] → submission_2hop_bpr128_only.csv  ({'OK' if r_bpr128.get('ok') else r_bpr128.get('message')})")

def blend_two(path_a: Path, path_b: Path, out_name: str) -> dict:
    if not path_a.exists() or not path_b.exists():
        return {"ok": False, "message": f"ファイル不足: {path_a.name} / {path_b.name}"}
    a = pd.read_csv(path_a)[["ID", "target"]].rename(columns={"target": "a"})
    b = pd.read_csv(path_b)[["ID", "target"]].rename(columns={"target": "b"})
    m = a.merge(b, on="ID")
    m["target"] = (m["a"] + m["b"]) / 2
    # ctx.test の ID 順に揃える（save_submission は行順で保存するため）
    m = m.set_index("ID").reindex(ctx.test["ID"]).reset_index()
    out = ctx.submissions_dir / out_name
    save_submission(ctx.test, m["target"].values, out, sanitize=True)
    return verify_submission(out, ctx.test)

# 4. BPR 64 と ALS 64 のブレンド
r_blend_als = blend_two(path_bpr64, ctx.submissions_dir / "submission_atmacup_implicit_als64.csv", "submission_blend_bpr64_als64.csv")
print(f"  [Blend BPR64+ALS64] → submission_blend_bpr64_als64.csv  ({'OK' if r_blend_als.get('ok') else r_blend_als.get('message')})")

# 5. BPR 64 と BPR 128 のブレンド
r_blend_bpr = blend_two(path_bpr64, ctx.submissions_dir / "submission_2hop_bpr128_only.csv", "submission_blend_bpr64_bpr128.csv")
print(f"  [Blend BPR64+BPR128] → submission_blend_bpr64_bpr128.csv  ({'OK' if r_blend_bpr.get('ok') else r_blend_bpr.get('message')})")

  0%|          | 0/100 [00:00<?, ?it/s]

  [BPR 128] → submission_2hop_bpr128_only.csv  (OK)
  [Blend BPR64+ALS64] → submission_blend_bpr64_als64.csv  (OK)
  [Blend BPR64+BPR128] → submission_blend_bpr64_bpr128.csv  (OK)


## 6. BPR 64 ベース スタッキング（LGB+XGB+CatBoost→Ridge）

In [7]:
from lib.improvement_candidates import get_bpr_base, run_03_stacking

# BPR 64 の特徴で ctx を差し替え（スタッキングは ctx.X / ctx.X_test を参照するため）
train_df, test_df, feats = get_bpr_base(ctx, factors=64)
for col in feats:
    if not pd.api.types.is_numeric_dtype(train_df[col]) and not pd.api.types.is_categorical_dtype(train_df[col]):
        train_df[col] = train_df[col].astype("category")
        test_df[col] = test_df[col].astype("category")
ctx_bpr = ctx._replace(X=train_df[feats], X_test=test_df[feats], features=feats)

# seed=None で submission_improvement_03_stacking.csv に保存（中身は乱数42）
r_stack = run_03_stacking(ctx_bpr, seed=None)
if r_stack.get("path"):
    print(f"  [BPR64 Stacking] → {r_stack['path'].name}  ({'OK' if r_stack.get('ok') else r_stack.get('message')})")
else:
    print(f"  [BPR64 Stacking] スキップ: {r_stack.get('message')}")

  0%|          | 0/100 [00:00<?, ?it/s]

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:rotten_tomatoes_link: category, critic_name: category, publisher_name: category, movie_title: category, movie_info: category, content_rating: category, directors: category, authors: category, actors: category, production_company: category

## 7. 劇的に上げる用（1位で「特徴量的に一番効いていた」）

「その映画に類似した映画をその批評家が何本レビューしているか」を 1 列追加。要 `outputs/embeddings/` の movie_title_info embedding（`train_baseline` や quick_embedding で生成済みならそのまま使える）。

In [ ]:
from lib.improvement_candidates import run_similar_movies_reviewed

r_sim = run_similar_movies_reviewed(ctx, top_k=20)
if r_sim.get("path"):
    print(f"  [類似映画を何本レビュー] → {r_sim['path'].name}  ({'OK' if r_sim.get('ok') else r_sim.get('message')})")
else:
    print(f"  [類似映画を何本レビュー] スキップ: {r_sim.get('message')}")

  0%|          | 0/100 [00:00<?, ?it/s]

## 提出ファイル一覧

In [ ]:
expected = [
    "submission_2hop_bpr64_only.csv",
    "submission_atmacup_implicit_als64.csv",
    "submission_2hop_bpr64_count1.csv",
    "submission_2hop_bpr64_ratio_count1.csv",
    "submission_2hop_bpr128_only.csv",
    "submission_blend_bpr64_als64.csv",
    "submission_blend_bpr64_bpr128.csv",
    "submission_improvement_03_stacking.csv",
    "submission_similar_movies_reviewed.csv",
]
for name in expected:
    p = ctx.submissions_dir / name
    print(f"  {'✓' if p.exists() else '✗'} {name}")

---

## まだやれること（このノートの外）

このノートで一通り出した**あと**に試せること。詳細は `docs/10_FIRST_PLACE_AND_IMPROVEMENT_REMAINING.md` と `docs/09_IMPROVEMENT_NEXT.md`。

| カテゴリ | 内容 | どこ |
|----------|------|------|
| **実装あり・BPR64に載せてない** | 類似度 TE（02）、擬似ラベル閾値見直し（01）、scale_pos_weight（05）、特徴選択（07）、弱点1列（10）、TF-IDF+SVD（08） | `lib.improvement_candidates` の run_02, run_01, run_05, run_07, run_10, run_08 を BPR64 の train/test で呼ぶ。別ノート or このノートにセル追加。 |
| **ブレンド・アンサンブル** | 重み付きブレンド（均等ではなく CV で重み探索）、シード平均（BPR64 を複数 seed で学習して予測を平均） | 09 に記載。低コスト。 |
| **協調フィルタ拡張** | confidence 重み（ALS）、NMF/TruncatedSVD、ALS ハイパーパラメータ、時系列で行列を切る | 09 §1。lib の `_build_implicit_embeddings` 拡張 or 新関数。中〜高コスト。 |
| **2-hop の修正版** | leave-one-out（リーク除去）、メタのみ 2-hop（target を使わない） | 10 §2.6。未実装。中コスト。 |
| **劇的に（未実装）** | メタの 2-hop、NN 1 本をアンサンブルに | 09 §4。新規実装。 |
| **評価・インフラ** | CV 加重（時系列+GroupKFold）でモデル選択、特徴量 feather 保存、wandb/hydra | 10 §2.5。低〜中コスト。 |